# 04 — GPU analysis (G1 + G3)

Throughput scaling vs batch size, and the 2×2 AMP × torch.compile grid, measured with CUDA events after warmup (compile time excluded).

**Storage contract** — code lives on GitHub, data & checkpoints live on Drive (`MyDrive/s2cd`), `/content` is disposable scratch. Every notebook is self-contained: run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
BRANCH = "chennai"          # the whole 00-07 series runs this branch
if os.path.exists(REPO):
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device, load_config

device = get_device()
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


In [ ]:
from s2cd.train import build_model
from s2cd.gpu.scale import batch_scaling, amp_compile_grid

cfg = load_config("configs/train.yaml")
model = build_model(cfg).to(device).eval()
shape = (1, cfg["in_channels"], 256, 256)  # (n, c, h, w)
print("params:", model.count_params())


In [ ]:
scale_rows = batch_scaling(model, shape, device,
                          batch_sizes=(8, 16, 32, 64), iters=20)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot([r["batch"] for r in scale_rows],
        [r["samples_per_sec"] for r in scale_rows], "o-")
ax.set_xlabel("batch size (images per forward pass)")
ax.set_ylabel("images processed per second")
ax.set_title("Batch size barely moves throughput -\nthe A100 is not the bottleneck")
ax.grid(True)
plt.tight_layout()
rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
fig.savefig(rep / "gpu_batch_scaling.png", dpi=150, bbox_inches="tight")
store.push_report("gpu_batch_scaling.png")
plt.show()


In [ ]:
# G3 precision/perf grid — fresh model per cell (compile is one-shot)
grid_rows = amp_compile_grid(lambda: build_model(cfg), shape, device,
                             batch=32, chip_hw=256, iters=20)
for r in grid_rows:
    print(r)


In [ ]:
# The 2x2 grid as a chart: each bar annotated with its speedup vs eager fp32.
# AMP + compile together beat the product of their individual gains -
# super-additive, because compile fuses the fp16 casts into the kernels.
import matplotlib.pyplot as plt

sps = {(r["amp"], r["compile"]): r["samples_per_sec"] for r in grid_rows
       if r["samples_per_sec"]}
labels = ["eager fp32\n(baseline)", "AMP only\n(fp16 math)",
          "compile only\n(kernel fusion)", "AMP + compile\n(both)"]
keys = [(False, False), (True, False), (False, True), (True, True)]
vals = [sps[k] for k in keys]
base = sps[(False, False)]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
bars = ax.bar(labels, vals, color="C0", width=0.6)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, v,
            f"{v:.0f} /s  ({v / base:.2f}x)", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("images processed per second")
ax.set_title("AMP x torch.compile (batch 32) - speedups vs eager fp32")
ax.set_ylim(0, max(vals) * 1.18)
plt.tight_layout()
rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
fig.savefig(rep / "amp_compile_grid.png", dpi=150, bbox_inches="tight")
store.push_report("amp_compile_grid.png")
plt.show()


In [ ]:
import json

rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)
(rep / "gpu_scaling.json").write_text(json.dumps(
    {"batch_scaling": scale_rows, "amp_compile_grid": grid_rows,
     "device": describe_device(device)}, indent=1))
store.push_report("gpu_scaling.json")
print("pushed gpu_scaling.json -> Drive")
